# Olist Analytics Copilot: eval run

Runs the golden set (120 questions) through the improved agent, **`semantic_plan`** (DECISIONS D27–D28): the model plans with governed metrics and code writes the SQL; questions no single metric answers fall back to custom SQL with a rule gate, result feedback and a 3-way majority vote. Every answer is scored against hand-written gold SQL.

Baseline from the first run (same model, same golden set, provisional): raw_schema 5%, semantic 42%, semantic_rag 60%. The three baseline modes are not re-run.

Questions run **4 at a time** against one vLLM server, so latency includes queueing.

**Provisional:** 20 of the gold answers are reviewed; the rest are provisional. The golden set also informed some of the improvements (D29), so treat it as a development set: a clean number needs a fresh held-out set.

| Step | Cell |
|---|---|
| Settings | 1 |
| Install the pinned serving stack and unpack the project code | 2 |
| Build the warehouse from the attached Olist data | 3 |
| Start the model server | 4 |
| Load the golden set and run every gold query | 5 |
| Ask every question in every mode | 6 |
| Headline results | 7 |
| Breakdowns by category | 8 |
| Failure gallery | 9 |
| Save results | 10 |

## 1. Settings
`LIMIT = None` runs all 120 items; set a number for a quick trial.

In [ ]:
MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"
MODES = ["semantic_plan"]  # the baseline modes are not re-run
LIMIT = None
CONCURRENCY = 4  # questions in flight at once; the T4 has room for ~20 full-length requests

## 2. Install and unpack
Pinned serving stack (DECISIONS D3). The project code comes from the private dataset `acme105/olist-copilot-code`, uploaded from a git commit.

In [ ]:
%pip install -q --no-cache-dir --retries 10 --timeout 120 "vllm==0.9.2" "transformers==4.53.2" "sqlglot>=25" "duckdb>=1.1" "pyyaml>=6.0"

import glob
import shutil
import sys
import tarfile
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
extracted = glob.glob("/kaggle/input/**/src/analytics_copilot/pipeline.py", recursive=True)
if extracted:
    shutil.copytree(Path(extracted[0]).parents[2], REPO, dirs_exist_ok=True)
else:
    with tarfile.open(glob.glob("/kaggle/input/**/copilot_code.tar.gz", recursive=True)[0]) as tar:
        tar.extractall(REPO)
sys.path.insert(0, str(REPO / "src"))
CODE_VERSION = (REPO / "VERSION").read_text().strip()
print("code version:", CODE_VERSION)

## 3. Build the warehouse
Raw CSVs → typed staging views → mart tables, exactly as locally (`make warehouse`).

In [ ]:
import duckdb

from analytics_copilot.warehouse import build_warehouse

data_dir = Path(glob.glob("/kaggle/input/**/olist_orders_dataset.csv", recursive=True)[0]).parent
WAREHOUSE = build_warehouse(data_dir, WORK / "olist.duckdb")
with duckdb.connect(str(WAREHOUSE), read_only=True) as con:
    for table in ["fct_orders", "fct_order_items", "fct_payments", "dim_seller_month"]:
        print(f"{table:<18} {con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]:>9,}")

## 4. Start the model server
vLLM serves the model as an OpenAI-compatible API on port 8000. fp16 on one T4.

In [ ]:
import os
import subprocess
import time

import requests

gpu = (
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True
    )
    .stdout.strip()
    .replace("\n", ", ")
)
print("GPU:", gpu)

server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        MODEL,
        "--dtype",
        "half",
        "--max-model-len",
        "8192",
        "--gpu-memory-utilization",
        "0.90",
        "--port",
        "8000",
    ],
    stdout=open(WORK / "vllm.log", "w"),
    stderr=subprocess.STDOUT,
    env={**os.environ, "VLLM_USE_V1": "0"},
)
deadline = time.time() + 20 * 60
while time.time() < deadline and server.poll() is None:
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            break
    except requests.ConnectionError:
        pass
    time.sleep(10)
else:
    print((WORK / "vllm.log").read_text()[-5000:])
    raise RuntimeError("vLLM did not start; log tail above")
# Warm-up: one real completion, so a server that loads but can't answer stops the run here
# (prefix caching loaded fine on the T4, then crashed on the first request: D32).
warmup = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json={"model": MODEL, "messages": [{"role": "user", "content": "Say OK."}], "max_tokens": 5},
    timeout=300,
)
if warmup.status_code != 200:
    print((WORK / "vllm.log").read_text()[-5000:])
    raise RuntimeError(f"vLLM loaded but cannot answer (HTTP {warmup.status_code}); log tail above")
print("model ready and answering:", MODEL)

## 5. Golden set and gold answers
Every gold query runs once against this warehouse. If any gold query fails, the run stops here.

In [ ]:
import pandas as pd

from analytics_copilot.evals.golden import load_golden
from analytics_copilot.evals.runner import run_gold

items = load_golden()[:LIMIT]
gold = await run_gold(items, WAREHOUSE)
print(
    f"{len(items)} items, {sum(i.verified for i in items)} verified, {len(gold)} gold queries ran"
)
pd.DataFrame(
    [{"difficulty": i.difficulty, "expected": i.expected_behaviour} for i in items]
).value_counts().to_frame("items")

## 6. Ask every question in every mode
Four questions at a time, so latency includes waiting for the shared GPU. Replies are recorded to `cache/llm.jsonl` so the run can be replayed and re-scored later without the GPU.

In [ ]:
from analytics_copilot.config import Settings
from analytics_copilot.evals.runner import CachingLLM, print_progress, run_eval
from analytics_copilot.llm import OpenAICompatibleClient
from analytics_copilot.pipeline import AskPipeline

settings = Settings(
    llm_base_url="http://localhost:8000/v1",
    sql_model=MODEL,
    summary_model=MODEL,
    warehouse_path=WAREHOUSE,
)
# replay=False: every call reaches the model (real latency); replies are only recorded.
llm = CachingLLM(
    OpenAICompatibleClient(settings), WORK / "cache" / "llm.jsonl", MODEL, replay=False
)
pipeline = AskPipeline.from_settings(settings, llm=llm)

started = time.time()
records = await run_eval(
    pipeline, items, MODES, gold, on_record=print_progress, concurrency=CONCURRENCY
)
print(f"\n{len(records)} answers in {(time.time() - started) / 60:.1f} min; cache hits: {llm.hits}")

## 7. Headline results

In [ ]:
from datetime import datetime, timezone

from IPython.display import Markdown, display

from analytics_copilot.evals.report import summarise, to_markdown

run = {
    "timestamp_utc": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
    "provider": "vllm 0.9.2 (OpenAI-compatible)",
    "model": MODEL,
    "quantisation": "none (fp16)",
    "hardware": gpu,
    "code_version": CODE_VERSION,
    "modes": MODES,
    "limit": LIMIT,
    "concurrency": CONCURRENCY,
    "notes": [
        f"{CONCURRENCY} questions in flight at once on one vLLM server; latency includes queueing.",
        "Golden set used as a development set (D29).",
    ],
    "cache_hits": llm.hits,
}
summary = summarise(records)
display(Markdown(to_markdown(run, summary)))

## 8. Accuracy by category

In [ ]:
rows = []
for mode, m in summary["all_items"].items():
    for category, a in m["by_category"].items():
        rows.append(
            {
                "mode": mode,
                "category": category,
                "accuracy": a["execution_accuracy"],
                "correct": a["correct"],
                "n": a["n"],
            }
        )
pd.DataFrame(rows).pivot(index="category", columns="mode", values="accuracy").style.format("{:.0%}")

## 9. Failure gallery
Every wrong answer: the question, the agent's SQL next to the gold SQL, both results, and the automatic failure label.

In [ ]:
wrong = [
    r for r in records if r["outcome"] in ("wrong", "error", "false_refusal", "missed_refusal")
]
print(f"{len(wrong)} failures\n")
for r in wrong:
    print("=" * 100)
    print(f"{r['id']} [{r['mode']}] {r['outcome']} ({r['failure_label']}) :: {r['question']}")
    if r["match_reason"]:
        print("why:", r["match_reason"])
    if r["error"]:
        print("error:", r["error"][:300])
    print("--- agent SQL ---\n", r["pred_sql"])
    print("--- gold SQL ---\n", (r["gold_sql"] or "(should refuse)").strip())
    print("agent rows:", r["pred_rows"][:5])
    print("gold rows: ", r["gold_rows"][:5])

## 10. Save results
Downloaded with `make kaggle-eval-results` and committed under `results/`.

In [ ]:
from analytics_copilot.evals.runner import write_results

server.terminate()
json_path, md_path, _ = write_results(run, records, WORK / "results")
print("wrote", json_path, "and", md_path)